# Perceptron implementation from scratch
### (Single weight update)

In [16]:
import numpy as np

In [17]:
#Activation Functions
def sigmoid(x):
  """1 / (1 + e^-x)"""
  return 1 / (1 + np.exp(-x))

def relu(x):
  """max(0, x)"""
  return np.maximum(0, x)

def tanh(x):
  return np.tanh(x)

def linear(x):
  return x

#Activation Derivatives
def sigmoid_derivative(x):
  """f(x) * (1 - f(x))"""
  return sigmoid(x) * (1 - sigmoid(x))

def relu_derivative(x):
  """1 if x > 0 else 0"""
  return np.where(x > 0, 1, 0)

def tanh_derivative(x):
  """1 - tanh^2(x)"""
  return 1 - np.tanh(x) ** 2

def linear_derivative(x):
  return 1

#Loss Functions
def mse(y_true, y_pred):
  return np.mean((y_true - y_pred) ** 2)

def mae(y_true, y_pred):
  return np.mean(np.abs(y_true - y_pred))

#Loss Derivatives (Supports both scalar and array inputs)
def mse_derivative(y_true, y_pred):
  """2 * (y_pred - y_true) / N"""
  n = np.size(y_true)
  return 2 * (y_pred - y_true) / n

def mae_derivative(y_true, y_pred):
  """sign(y_pred - y_true) / N"""
  n = np.size(y_true)
  return np.sign(y_pred - y_true) / n

In [33]:
class Neuron:
  # Class attributes placeholders
  input_matrix = np.array([])
  weight_vector = np.array([])
  target = 0
  bias = 0
  input_length = 0
  dataset_length = 0
  learning_rate = 0

  activation_function = None
  loss_function = None
  activation_derivative = None
  loss_derivative = None

  # Mapping functions
  activation_functions = {
    "sigmoid": sigmoid,
    "relu": relu,
    "tanh": tanh,
    "linear": linear,
  }
  activation_derivatives = {
    "sigmoid": sigmoid_derivative,
    "relu": relu_derivative,
    "tanh": tanh_derivative,
    "linear": linear_derivative,
  }
  loss_functions = {
    "mse": mse,
    "mae": mae,
  }
  loss_derivatives = {
    "mse": mse_derivative,
    "mae": mae_derivative,
  }

  #Constructor
  def __init__(self, input_matrix, target, activation_function, error_function, learning_rate=0.1, weight_vector=None, bias=0):
    """Initializes the Neuron instance and maps functions."""
    self.input_matrix = np.array(input_matrix)
    self.input_length = len(self.input_matrix[0]) if len(self.input_matrix) > 0 else 0
    self.target = np.array(target)
    self.dataset_length = len(input_matrix)

    self.activation_function = self.activation_functions[activation_function]
    self.activation_derivative = self.activation_derivatives[activation_function]
    self.learning_rate = learning_rate
    self.loss_derivative = self.loss_derivatives[error_function]
    self.loss_function = self.loss_functions[error_function]

    # Weight initialization (defaults to zeros if none provided)
    if weight_vector is None:
      self.weight_vector = np.zeros(self.input_length)
    else:
      self.weight_vector = np.array(weight_vector)
    self.bias = bias

  def forward_pass(self, current_index):
    """Calculates z = W.X + b, and returns f(z)."""
    x = self.input_matrix[current_index]
    z = np.dot(x, self.weight_vector) + self.bias
    return self.activation_function(z)

  def error(self, current_index):
    """Calculates loss for a single sample."""
    return self.loss_function(self.target[current_index], self.forward_pass(current_index))

  def update_weights(self, current_index):
    """Performs single-sample stochastic gradient descent weight and bias updates."""
    x = self.input_matrix[current_index]
    y_true = self.target[current_index]

    # Forward propagation step
    z = np.dot(x, self.weight_vector) + self.bias
    y_pred = self.activation_function(z)

    # Chain rule parts: dLoss/dy_pred and dy_pred/dz
    dloss = self.loss_derivative(y_true, y_pred)
    dactivation = self.activation_derivative(z)

    # Overall scalar gradient at the linear combination output
    gradient = dloss * dactivation

    # Parameter updates (multiply by input x only for weights)
    self.weight_vector -= self.learning_rate * gradient * x
    self.bias -= self.learning_rate * gradient

### Testing Perceptron on AND and OR gates

In [34]:
# 1. AND Gate Dataset
and_inputs = [[0, 0], [0, 1], [1, 0], [1, 1]]
and_targets = [0, 0, 0, 1]

# Instantiate Neuron for AND Gate
and_neuron = Neuron(
    input_matrix=and_inputs,
    target=and_targets,
    activation_function='sigmoid',
    error_function='mse',
    learning_rate=0.5,
)

# Training
epochs = 70
for epoch in range(epochs):
    for i in range(len(and_inputs)):
        and_neuron.update_weights(i)

# Predict and display results
print("AND Gate Predictions\n")
for i in range(len(and_inputs)):
    pred = and_neuron.forward_pass(i)
    print(f"Input: {and_inputs[i]} -> Pred: {pred:.4f} (Class: {1 if pred >= 0.5 else 0}) | Target: {and_targets[i]}")
print("\nFinal Weights:", and_neuron.weight_vector)
print("Final Bias:", and_neuron.bias)

AND Gate Predictions

Input: [0, 0] -> Pred: 0.0287 (Class: 0) | Target: 0
Input: [0, 1] -> Pred: 0.2264 (Class: 0) | Target: 0
Input: [1, 0] -> Pred: 0.2350 (Class: 0) | Target: 0
Input: [1, 1] -> Pred: 0.7525 (Class: 1) | Target: 1

Final Weights: [2.34072259 2.29209598]
Final Bias: -3.5207739159994467


In [35]:
# 2. OR Gate Dataset
or_inputs = [[0, 0], [0, 1], [1, 0], [1, 1]]
or_targets = [0, 1, 1, 1]

# Instantiate Neuron for OR Gate
or_neuron = Neuron(
    input_matrix=or_inputs,
    target=or_targets,
    activation_function='sigmoid',
    error_function='mse',
    learning_rate=0.5,
)

# Training
epochs = 70
for epoch in range(epochs):
    for i in range(len(or_inputs)):
        or_neuron.update_weights(i)

# Predict and display results
print("OR Gate Predictions\n")
for i in range(len(or_inputs)):
    pred = or_neuron.forward_pass(i)
    print(f"Input: {or_inputs[i]} -> Pred: {pred:.4f} (Class: {1 if pred >= 0.5 else 0}) | Target: {or_targets[i]}")
print("\nFinal Weights:", or_neuron.weight_vector)
print("Final Bias:", or_neuron.bias)

OR Gate Predictions

Input: [0, 0] -> Pred: 0.2531 (Class: 0) | Target: 0
Input: [0, 1] -> Pred: 0.8592 (Class: 1) | Target: 1
Input: [1, 0] -> Pred: 0.8560 (Class: 1) | Target: 1
Input: [1, 1] -> Pred: 0.9907 (Class: 1) | Target: 1

Final Weights: [2.86429915 2.8910802 ]
Final Bias: -1.0821894453107872
